# Snippet from Math-Lyapunov-Stability.md


In [ ]:
class TaskSequencer:
    """Lyapunov-based adaptive task difficulty manager."""
    
    def __init__(self):
        self.controller = LyapunovController(initial_radius=1.0)
        self.student_load = 0.5  # Initial cognitive load
        
    def assess_load(self, task_complexity: float, student_state: dict) -> float:
        """Estimate cognitive load for proposed task."""
        base_load = task_complexity
        fatigue_factor = student_state.get('fatigue', 0)
        confidence_bonus = student_state.get('confidence', 0.5)
        
        return base_load * (1 + fatigue_factor) * (2 - confidence_bonus)
    
    def propose_task(self, task_pool: list, student_state: dict) -> dict:
        """Select next task using Lyapunov stability criterion."""
        current_load = self.student_load
        
        for task in sorted(task_pool, key=lambda t: t['complexity']):
            proposed_load = self.assess_load(task['complexity'], student_state)
            
            r, accept, metrics = self.controller.lyapunov_gate(
                current_load, 
                proposed_load
            )
            
            if accept:
                self.student_load = proposed_load
                return {
                    'task': task,
                    'expected_load': proposed_load,
                    'gate_decision': metrics
                }
        
        # If all rejected, return recovery task
        return {
            'task': {'name': 'Review', 'complexity': 0.1},
            'expected_load': 0.1,
            'gate_decision': {'status': 'RECOVERY'}
        }

# Usage
sequencer = TaskSequencer()
tasks = [
    {'name': 'Basic Algebra', 'complexity': 0.3},
    {'name': 'Word Problems', 'complexity': 0.6},
    {'name': 'Proof Writing', 'complexity': 0.9}
]
student = {'fatigue': 0.2, 'confidence': 0.7}

next_task = sequencer.propose_task(tasks, student)
print(f"Assigned: {next_task['task']['name']}")
print(f"Expected load: {next_task['expected_load']:.2f}")
